In [ ]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry
from pathlib import Path

# 1. Setup Open-Meteo API client
cache_session = requests_cache.CachedSession(".cache", expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# 2. API request settings
url = "https://historical-forecast-api.open-meteo.com/v1/forecast"

params = {
    "latitude": -38.1387,
    "longitude": 176.2452,
    "start_date": "2017-01-01",
    "end_date": "2026-06-06",
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "rain",
        "pressure_msl",
        "surface_pressure",
        "cloud_cover",
        "wind_speed_10m",
        "wind_gusts_10m",
        "temperature_80m",
        "wind_direction_10m",
        "shortwave_radiation"
    ],
    "wind_speed_unit": "ms"
}

responses = openmeteo.weather_api(url, params=params)
response = responses[0]
hourly = response.Hourly()

# 3. Create timestamps
time_utc = pd.date_range(
    start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
    end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
    freq=pd.Timedelta(seconds=hourly.Interval()),
    inclusive="left"
)

time_nz = time_utc.tz_convert("Pacific/Auckland")

# 4. Build dataframe
hourly_dataframe = pd.DataFrame({
    "time_utc": time_utc,
    "time_nz": time_nz,
    "time_nz_clean": time_nz.tz_localize(None),

    "temperature_2m_C": hourly.Variables(0).ValuesAsNumpy(),
    "relative_humidity_2m_percent": hourly.Variables(1).ValuesAsNumpy(),
    "precipitation_mm": hourly.Variables(2).ValuesAsNumpy(),
    "rain_mm": hourly.Variables(3).ValuesAsNumpy(),
    "pressure_msl_hPa": hourly.Variables(4).ValuesAsNumpy(),
    "surface_pressure_hPa": hourly.Variables(5).ValuesAsNumpy(),
    "cloud_cover_percent": hourly.Variables(6).ValuesAsNumpy(),
    "wind_speed_10m_ms": hourly.Variables(7).ValuesAsNumpy(),
    "wind_gusts_10m_ms": hourly.Variables(8).ValuesAsNumpy(),
    "temperature_80m_C": hourly.Variables(9).ValuesAsNumpy(),
    "wind_direction_10m_deg": hourly.Variables(10).ValuesAsNumpy(),
    "shortwave_radiation_W_m2": hourly.Variables(11).ValuesAsNumpy()
})

# 5. Basic checks
print(hourly_dataframe.head())
print(hourly_dataframe.tail())

print("\nDate range:")
print("NZ start:", hourly_dataframe["time_nz"].min())
print("NZ end:", hourly_dataframe["time_nz"].max())

print("\nMissing values:")
print(hourly_dataframe.isna().sum())

print("\nShortwave radiation range:")
print(hourly_dataframe["shortwave_radiation_W_m2"].min())
print(hourly_dataframe["shortwave_radiation_W_m2"].max())

# 6. Save file
output_path = Path(
    r"C:\Users\sh1345\code\myproject\Project 1\data\weather data\open_meteo_historical_forecast_Rotorua_2017_2026.csv"
)

hourly_dataframe.to_csv(output_path, index=False)

print(f"\nSaved to: {output_path}")

                   time_utc                   time_nz       time_nz_clean  \
0 2017-01-01 00:00:00+00:00 2017-01-01 13:00:00+13:00 2017-01-01 13:00:00   
1 2017-01-01 01:00:00+00:00 2017-01-01 14:00:00+13:00 2017-01-01 14:00:00   
2 2017-01-01 02:00:00+00:00 2017-01-01 15:00:00+13:00 2017-01-01 15:00:00   
3 2017-01-01 03:00:00+00:00 2017-01-01 16:00:00+13:00 2017-01-01 16:00:00   
4 2017-01-01 04:00:00+00:00 2017-01-01 17:00:00+13:00 2017-01-01 17:00:00   

   temperature_2m_C  relative_humidity_2m_percent  precipitation_mm  rain_mm  \
0         19.934502                     54.489464               0.0      0.0   
1         18.634501                     69.000114               0.2      0.2   
2         20.284500                     62.069958               0.1      0.1   
3         19.884501                     63.209991               0.0      0.0   
4         20.034500                     63.243530               0.0      0.0   

   pressure_msl_hPa  surface_pressure_hPa  cloud_cover_p

In [ ]:
print(hourly_dataframe["time_nz"].min())
print(hourly_dataframe["time_nz"].max())
print(hourly_dataframe.isna().sum())
print(hourly_dataframe.describe())

In [ ]:
hourly_dataframe["year"] = hourly_dataframe["time_nz"].dt.year

print(
    hourly_dataframe.groupby("year")["shortwave_radiation_W_m2"]
    .agg(["count", "min", "max", "mean"])
)